# Diagnostic Percept — 01 · Discovery (H1–H5)

Find the diagnosis-gate neuron (H1), disease concept neurons (H2), the symptom→diagnosis routing layer (H3), hallucination neurons (H4 + per-category), and overconfidence neurons (H5). Cheap (~20–30 min on one A100). Writes `results/discovery.json` for later phases.

**Cross-phase state** is shared through `results/` mirrored to a GCS bucket (set `GCS_BUCKET`) or Google Drive. Run the phases in order: `00 → 01 → 02 → 03 → 04`. Each is a *separate* Colab Enterprise runtime; the persistence cells restore the previous phase's outputs.

Models are **Qwen3 only** (32B → 14B → 8B → 4B auto-picked by GPU memory); no Med42/Med43 anywhere.

## 1. Setup — install, GPU check, clone, HF login

In [ ]:
# Boot disk on Vertex AI Colab Enterprise is ~101 GB and starts ~60 GB
# full (system image). /content is the 527 GB workspace. Without
# redirection, pip's temp build files + pip cache + HF cache all land
# on the boot disk and can fill it during the install — at which point
# Vertex AI health checks fail and the runtime is marked unhealthy.
# Set EVERY cache dir to /content BEFORE the first pip call.
import os, subprocess, sys, shutil
from pathlib import Path
_C = Path('/content/.cache') if Path('/content').exists() else None
if _C:
    _C.mkdir(parents=True, exist_ok=True)
    (_C / 'pip').mkdir(exist_ok=True)
    (_C / 'tmp').mkdir(exist_ok=True)
    os.environ['PIP_CACHE_DIR']  = str(_C / 'pip')
    os.environ['TMPDIR']         = str(_C / 'tmp')
    os.environ['HF_HOME']        = str(_C / 'huggingface')
    os.environ['HF_HUB_CACHE']   = str(_C / 'huggingface')
    os.environ['TRANSFORMERS_CACHE'] = str(_C / 'transformers')
    os.environ['TORCH_HOME']     = str(_C / 'torch')
    os.environ['XDG_CACHE_HOME'] = str(_C)
    print(f'Caches → {_C} (boot disk is small; this is mandatory)')

def _disk(label=''):
    for p in ('/', '/content'):
        if Path(p).exists():
            s = shutil.disk_usage(p)
            free = (s.total - s.used) / 1e9
            print(f'  [{label}] disk {p:<10} free={free:6.1f} GB')
_disk('start')

# Surgical upgrade: install transformers main with --no-deps so it does
# NOT pull a newer torch / torchvision / pillow. Then pin transformers'
# runtime deps to the *exact* versions it expects (it pins tokenizers
# <=0.23.0, which a bare `--upgrade tokenizers` overshoots to 0.23.1).
def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

# 1. transformers main, no cascading dep upgrades
_pip('--upgrade', '--no-deps',
     'transformers @ git+https://github.com/huggingface/transformers.git@main')
# 2. transformers' runtime deps. huggingface_hub MUST come from main
#    too — transformers main imports `is_offline_mode` which only
#    exists in hub's main branch (older released versions removed it,
#    newer renamed it). Install hub from git@main to match.
_pip('--no-deps', '--upgrade',
     'huggingface_hub @ git+https://github.com/huggingface/huggingface_hub.git@main',
     'safetensors>=0.4',
     'tokenizers>=0.22.0,<=0.23.0',
     'regex',
     'requests',
     'pyyaml',
     'httpx',
     'filelock')
# 3. our other libs --no-deps (accelerate / bitsandbytes happy w/ Colab torch)
_pip('--upgrade', '--no-deps', 'accelerate>=0.34', 'bitsandbytes>=0.43')
# 4. plain installs of small libs (no risk to torch/pillow)
_pip('scikit-learn', 'matplotlib', 'tqdm', 'datasets', 'nbformat', 'ipywidgets')
_disk('after step 4')
# 5. Pillow self-heal if a prior run pulled pillow 12 (PIL.ImageText breaks).
try:
    import PIL.ImageText  # canary for pillow 12 ABI break
except Exception:
    print('Repairing pillow (pinning <12) ...')
    _pip('--force-reinstall', '--no-deps', 'pillow<12')

# Free pip cache to reclaim disk now that everything is installed.
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False, capture_output=True)
_disk('after purge')

# Drop the pre-imported transformers + huggingface_hub from Colab so the
# re-import picks up the new versions.
import importlib
for m in [k for k in list(sys.modules)
          if k in ('transformers', 'huggingface_hub')
          or k.startswith('transformers.') or k.startswith('huggingface_hub.')]:
    del sys.modules[m]
importlib.invalidate_caches()

# Self-heal: if the import still fails because hub<->transformers got
# out of sync, re-install both from main and retry once.
try:
    import transformers
except ImportError as _e:
    print(f'Self-healing transformers/hub mismatch: {_e}')
    _pip('--no-deps', '--upgrade', '--force-reinstall',
         'transformers @ git+https://github.com/huggingface/transformers.git@main',
         'huggingface_hub @ git+https://github.com/huggingface/huggingface_hub.git@main')
    for m in [k for k in list(sys.modules)
              if k in ('transformers', 'huggingface_hub')
              or k.startswith('transformers.') or k.startswith('huggingface_hub.')]:
        del sys.modules[m]
    importlib.invalidate_caches()
    import transformers
_has_q35 = hasattr(transformers, 'Qwen3_5ForCausalLM')
print(f'transformers {transformers.__version__}  Qwen3_5 registered: {_has_q35}')
if not _has_q35:
    # NB: do NOT auto-restart the kernel here. Vertex AI's idle detector
    # interprets the post-restart wait as inactivity and may shut the VM
    # down within minutes. Instead, halt cleanly with a clear message so
    # the user does the restart manually and immediately Run All again.
    raise SystemExit(
        '\n' + '=' * 70 +
        '\n  ACTION REQUIRED: restart the kernel, then click Run All again.'
        '\n  Colab Enterprise: Runtime → Restart session → Run all.'
        '\n  (Auto-restart removed because Vertex AI counts the post-'
        '\n   restart idle time toward the auto-shutdown timer.)'
        '\n' + '=' * 70
    )

In [ ]:
# === EMERGENCY DISK CLEANUP — uncomment, run, re-comment ====================
# Use when `df -h /` shows < 10 GB free on the boot disk after the install
# step. Each block is independent; you can run just one or all of them.
#
# import subprocess, shutil, gc, os
# from pathlib import Path
#
# # 1. Boot-disk caches (the usual offenders).
# for d in ('/root/.cache/pip', '/root/.cache/huggingface',
#           '/root/.cache/torch', '/root/.cache/matplotlib',
#           '/root/.cache/black', '/root/.triton'):
#     subprocess.run(['rm', '-rf', d], check=False)
# # 2. /tmp leftovers (pip build dirs, torch inductor, model shards).
# for pattern in ('/tmp/pip*', '/tmp/torch*', '/tmp/cuda*', '/tmp/hf*'):
#     subprocess.run(f'rm -rf {pattern}', shell=True, check=False)
# # 3. The pip download cache (~/ + system).
# subprocess.run(['python', '-m', 'pip', 'cache', 'purge'], check=False)
# # 4. Stale HuggingFace lockfiles on the workspace (rare; safe to clear).
# subprocess.run(['find', '/content/.cache/huggingface', '-name', '*.lock',
#                 '-delete'], check=False)
# # 5. Old results from a prior run on /content (only if you don't need them).
# # shutil.rmtree('/content/results', ignore_errors=True)
# # 6. CUDA allocator + Python garbage. Releases any held GPU mem.
# gc.collect()
# try:
#     import torch
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         for i in range(torch.cuda.device_count()):
#             torch.cuda.reset_peak_memory_stats(i)
# except Exception:
#     pass
# for p in ('/', '/content'):
#     if Path(p).exists():
#         s = shutil.disk_usage(p)
#         print(f'  disk {p:<10} free={(s.total-s.used)/1e9:6.1f} GB')
# ============================================================================

In [ ]:
import os
from pathlib import Path
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# Redirect HF / Torch caches to /content (Colab Enterprise's 195 GB workspace
# disk) so model weights don't fill the ~90 GB boot disk. Must happen before
# transformers / huggingface_hub are imported, so set it here.
_CACHE_ROOT = '/content/.cache' if Path('/content').exists() else None
if _CACHE_ROOT:
    Path(_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME',          f'{_CACHE_ROOT}/huggingface')
    os.environ.setdefault('TRANSFORMERS_CACHE', f'{_CACHE_ROOT}/transformers')
    os.environ.setdefault('TORCH_HOME',       f'{_CACHE_ROOT}/torch')
    os.environ.setdefault('XDG_CACHE_HOME',   _CACHE_ROOT)
    print(f'Caches redirected to {_CACHE_ROOT}')
else:
    print('No /content workspace (not on Colab); using default cache dirs.')

# Force tqdm.notebook so progress bars render as Colab widgets, not raw lines
# (matters for the long H6/H7/sycophancy passes).
try:
    import tqdm, tqdm.notebook
    tqdm.tqdm = tqdm.notebook.tqdm
    import tqdm.auto
    tqdm.auto.tqdm = tqdm.notebook.tqdm
    print('tqdm.notebook installed as the default tqdm')
except Exception as _e:
    print('tqdm.notebook unavailable, keeping default:', _e)

In [ ]:
# === env check ===
import traceback
try:

    import os, sys, subprocess, json, time, traceback, importlib
    from pathlib import Path
    import torch

    # Runtime detection — free Colab vs Colab Enterprise (Vertex Workbench) vs other.
    def _detect_runtime():
        if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
            try:
                import google.colab  # noqa: F401
                return 'colab_free'
            except ImportError:
                pass
        if any(k in os.environ for k in ('GOOGLE_CLOUD_PROJECT', 'VERTEX_PRODUCT')):
            return 'colab_enterprise'
        if 'JUPYTERHUB_USER' in os.environ:
            return 'jupyterhub'
        return 'local'
    RUNTIME = _detect_runtime()
    print(f'Runtime: {RUNTIME}')
    print('Python:', sys.version.split()[0])
    print('Torch :', torch.__version__)
    print('CUDA  :', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        gpu_name = torch.cuda.get_device_name(0)
        print(f'GPU: {gpu_name}  | Memory: {gpu_gb:.1f} GB')

        # NOTE: do NOT set MODEL_OVERRIDE / USE_4BIT / N_BENCH here. All
        # decisioning lives in src/setup.py auto_pick(), which is called by
        # smart_load_model() in the model-load cell. If you set env vars
        # here, an old snapshot of THIS cell (frozen in your imported .ipynb)
        # could write a stale Qwen3.5/3.6 pick that auto_pick can't override
        # because the env var "wins". src/setup.py also actively strips
        # MODEL_OVERRIDE if it points to a known-broken Qwen3.5/3.6 checkpoint.

    # Disk sanity. Colab Enterprise's boot disk is ~94 GB and starts ~90% full
    # (system image). /content is the 195 GB workspace where caches go.
    import shutil
    for path in ('/', '/content'):
        if Path(path).exists():
            s = shutil.disk_usage(path)
            used_pct = 100 * s.used / s.total
            warn = ' !! LOW' if (s.total - s.used) < 10 * (1024**3) else ''
            print(f'Disk {path:<10}  {s.used/1e9:6.1f} / {s.total/1e9:6.1f} GB  ({used_pct:.0f}%){warn}')

    # Validate cache redirect — the model download (~14 GB at NF4, ~54 GB at bf16)
    # MUST land on /content or the boot disk fills up.
    _hf_home = os.environ.get('HF_HOME', '')
    if _hf_home and not _hf_home.startswith('/content'):
        print('!! WARN: HF_HOME is', _hf_home, '— model will download to boot disk!')
    elif _hf_home:
        print(f'HF cache → {_hf_home}  (/content has plenty of room)')
    else:
        print('!! WARN: HF_HOME not set; model download will use ~/.cache (boot disk).')

    REPO_URL = 'https://github.com/ArioMoniri/diagnosticpercept.git'
    REPO_DIR = 'diagnosticpercept'
    if not Path(REPO_DIR).exists():
        print('Cloning', REPO_URL, '...')
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        # Hard-reset to origin/main so re-runs always pick up the latest code.
        print('Fetching + hard-resetting to origin/main ...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=False)

    # Print current SHA so we can verify the running version.
    sha = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print(f'Repo @ commit: {sha}  (expect 8635794 or newer for H4+H5)')

    # Drop any previously-imported src.* modules so Python re-loads from disk —
    # a kernel re-run with the prior clone may have cached the old discover.py.
    for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
        del sys.modules[m]
    importlib.invalidate_caches()

    repo_path = str(Path(REPO_DIR).resolve())
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    RESULTS = Path('/content/results'); RESULTS.mkdir(parents=True, exist_ok=True)
    print('Repo   :', repo_path)
    print('Results:', RESULTS)
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

### 🚀 4× A100 quickstart

This notebook detects multiple GPUs automatically. On a `a2-highgpu-4g` (4× A100 40 GB) or `a2-ultragpu-4g` (4× A100 80 GB) VM the H6 benchmark runs in **data-parallel mode** — one model copy per GPU, ~4× wall-clock speedup. NF4 quantization is forced in the workers (~14 GB / GPU after load), so the full Qwen3-32B + the H6 reasoning-chain KV cache + activations all fit on a single A100-40.

**Memory budget per worker (NF4, Qwen3-32B):**

| Item | A100-40 | A100-80 |
|------|--------:|--------:|
| Model weights (NF4) | ~14 GB | ~14 GB |
| Reasoning KV cache (512 new tok) | ~6 GB | ~6 GB |
| Per-step activations + scores | ~3 GB | ~3 GB |
| Safety headroom | ~17 GB | ~57 GB |
| **Per-GPU peak under H6** | **~23 GB / 40** | **~23 GB / 80** |

**Disk budget (Colab Enterprise):** boot disk is ~94 GB and starts ~90 % full. The model download (~14 GB at NF4) **must** land on `/content` (195 GB workspace). The Section 1 install cell already redirects every cache to `/content/.cache/` before the first `pip install`, so under normal operation the boot disk stays at its starting level.

If a previous run left the boot disk full anyway, uncomment the *EMERGENCY DISK CLEANUP* cell above, run it once, then re-comment.

**Estimated wall time on 4× A100-40 G**, NF4, full pipeline:
H1 ~5 min · H2 ~3 min · H3 ~30 min · H4 ~5 min · H5 ~5 min · H6 (1273 × 6 conditions, DP) ~50 min · H7 ~6 min · H8 ~10 min · sycophancy ~15 min  ⇒  **~2 h 10 min end-to-end.**

In [ ]:
# === preflight — print the run plan ===
import traceback
try:

    # Single-glance summary of what's about to happen so you can abort before
    # downloading 14 GB of model weights if anything is wrong.
    print('=' * 62)
    print(f'  Runtime       : {RUNTIME}')
    _n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if _n_gpu == 0:
        print('  GPU           : none (CPU only)')
    else:
        # PER-GPU memory report — important on 4× A100 because nvidia-smi may
        # show a heterogeneous mix if one GPU was previously used by another
        # process (Vertex AI doesn't always reset cleanly across notebook runs).
        for i in range(_n_gpu):
            p = torch.cuda.get_device_properties(i)
            gb = p.total_memory / 1e9
            used = torch.cuda.memory_allocated(i) / 1e9
            print(f'  GPU{i}          : {p.name}  total={gb:.1f} GB  '
                  f'currently_allocated={used:.2f} GB')
    print(f'  Model         : {os.environ.get("MODEL_OVERRIDE", "(auto-pick from chain)")}')
    print(f'  Quantize 4bit : {os.environ.get("USE_4BIT", "auto")}')
    print(f'  N_BENCH       : {os.environ.get("N_BENCH", "default")}')
    print(f'  HF cache      : {os.environ.get("HF_HOME", "(default ~/.cache)")}')
    print(f'  CUDA alloc    : {os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "(unset)")}')
    print()
    print('  Estimated wall time on this hardware:')
    _gpu_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if _n_gpu else 0
    _gpu_name = torch.cuda.get_device_name(0) if _n_gpu else ''
    # H100 ≈ 1.5× A100 fwd throughput. Wall time scales by 1/n_gpu for H6.
    _is_h100 = 'H100' in _gpu_name
    _throughput_factor = 1.0 if _is_h100 else 1.5  # A100 vs H100
    _par = max(1, _n_gpu)
    print(f'  Hardware: {_n_gpu}× {_gpu_name or "CPU"}  (parallel factor {_par})')
    if _gpu_gb >= 36:
        h1   = round(5  * _throughput_factor, 1)             # H1 stays single-GPU
        h6   = round(75 * _throughput_factor / _par, 1)      # parallelized
        h7   = round(6  * _throughput_factor, 1)             # single-GPU
        syc  = round(15 * _throughput_factor, 1)             # single-GPU for now
        total = h1 + h6 + h7 + syc
        print(f'    H1 discover           ~{h1} min  (single GPU)')
        print(f'    H6 deep (1273×6)      ~{h6} min  ({_par}× parallel)')
        print(f'    H7 (300 items)        ~{h7} min  (single GPU)')
        print(f'    H8 + sycophancy       ~{syc} min  (single GPU)')
        print(f'    --- TOTAL             ~{total/60:.1f} hr')
    else:
        print('    Small-GPU budget — auto-pick will drop model size.')
    print('=' * 62)

    # On 4-GPU machines the data-parallel H6 worker holds an extra ~3 GB cuBLAS
    # workspace per device by default. Setting CUBLAS_WORKSPACE_CONFIG=:0:0
    # disables that pool (we don't need deterministic cuBLAS for inference) and
    # saves ~12 GB across 4 GPUs — buys back the H6 KV cache headroom on A100-40.
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':0:0')

    # expandable_segments cuts fragmentation across the many small allocs the
    # H6 reasoning chain produces (every gen.scores entry is its own alloc).
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === HF login (optional — only for gated models) ===
import traceback
try:

    import os
    # Qwen3 is open-weights and needs NO token. HF_TOKEN is only needed if you
    # override to a gated model. Resolution order:
    #   1. env var HF_TOKEN
    #   2. Colab secret HF_TOKEN
    #   3. interactive notebook_login() widget (may not render in all Colab
    #      runtimes — if so, use the manual paste cell that follows)
    def _resolve_hf_token():
        if os.environ.get('HF_TOKEN'):
            print('HF_TOKEN already set in env.')
            return
        # Free Colab has google.colab.userdata; Colab Enterprise does NOT.
        if RUNTIME == 'colab_free':
            try:
                from google.colab import userdata
                tok = userdata.get('HF_TOKEN')
                if tok:
                    os.environ['HF_TOKEN'] = tok
                    print('HF_TOKEN loaded from Colab secret.')
                    return
            except Exception:
                pass
        print('No HF_TOKEN in env.')
        if RUNTIME == 'colab_enterprise':
            print('Colab Enterprise: set HF_TOKEN as a runtime-template env var,')
            print('or paste into the manual cell below.')
        else:
            print('Qwen3 is open-weights so this is fine to skip for the default chain.')
        try:
            from huggingface_hub import notebook_login
            notebook_login()
            print('Token widget rendered above ↑ (paste + Login).')
            print('If you do not see a widget, use the manual paste cell below.')
        except Exception as e:
            print(f'(notebook_login unavailable: {e})')

    _resolve_hf_token()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === HF token: manual paste fallback (skip if widget worked) ===
import traceback
try:

    # If the widget above didn't render, paste your token below between the quotes
    # and run THIS cell. Leave blank to skip.
    HF_TOKEN_PASTE = ''   # ← paste like 'hf_xxxxxxxxxxxxxxxxx', then Run cell

    if HF_TOKEN_PASTE.strip():
        os.environ['HF_TOKEN'] = HF_TOKEN_PASTE.strip()
        print(f'HF_TOKEN set manually ({len(HF_TOKEN_PASTE.strip())} chars).')
    else:
        print('No manual token pasted. Continuing with whatever the previous cell resolved.')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 2. Load model + verify hooks

Default chain is **Qwen-only**, Qwen3 family (Qwen3-32B → 14B → 8B → 4B). The first that fits VRAM (with NF4 4-bit below 48 GB) wins. Qwen3.5/3.6 are not on the chain — their checkpoints don't load cleanly on transformers' Qwen3_5ForCausalLM class today. Patches every MLP forward to expose `h = SiLU(W_gate x) * (W_up x)` with `retain_grad`.

*To force a specific Qwen variant*: set `os.environ['MODEL_OVERRIDE'] = 'Qwen/<exact-repo-name>'` **before** running this cell.

In [ ]:
# === load model ===
import traceback
try:

    # All decision logic lives in src/setup.py — fixes to GPU detection, model
    # auto-pick, or max_memory take effect on the next Run All without
    # re-importing the notebook (the env-check cell pulls latest src/ first).
    from src.setup import smart_load_model
    from src.model import set_seed
    set_seed(0)

    lm, MODEL_NAME = smart_load_model()
    # Legacy globals so downstream cells keep working.
    USE_4BIT = bool(int(os.environ.get('USE_4BIT', '0')))
    N_BENCH  = int(os.environ.get('N_BENCH', '1273'))
    n_gpus   = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === memory helpers — reclaim VRAM + disk between sections ===
import traceback
try:

    # Lightweight helpers we'll call between H1/H2/.../H8 to keep VRAM bounded.
    # H4-H7 each cache large activation tensors in Python globals; without an
    # explicit drop between sections the cuBLAS allocator's reserved pool
    # ratchets up and the H6 reasoning chain can OOM 90 min in.
    import gc, shutil
    from pathlib import Path

    def _free_vram(label=''):
        gc.collect()
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats(i)
            used = [torch.cuda.memory_allocated(i)/1e9
                    for i in range(torch.cuda.device_count())]
            print(f'  [free_vram {label}] alloc/GPU = ' +
                  ' '.join(f'{u:.2f}' for u in used) + ' GB')
        # /content disk free.
        if Path('/content').exists():
            s = shutil.disk_usage('/content')
            print(f'  [free_vram {label}] /content free = {(s.total-s.used)/1e9:.1f} GB')

    _free_vram('post-load')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === sanity: h.retain_grad flows ===
import traceback
try:

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    ids = lm.tokenizer('Chest pain. Diagnosis:', return_tensors='pt').input_ids.to(lm.device)
    lm.model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        out = lm.model(input_ids=ids, use_cache=False)
        # logit at last position only — no need for full vocab sum.
        out.logits[0, -1, 0].backward()
    g = lm.layers[0].mlp._h.grad
    assert g is not None, 'h.grad is None — hook patching failed.'
    assert torch.isfinite(g).all(), 'h.grad has non-finite values.'
    assert g.abs().sum() > 0, 'h.grad is all zeros.'
    print('OK: layer-0 h.grad shape', tuple(g.shape), 'nonzero =', (g.abs() > 0).sum().item())
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'  VRAM after sanity: {torch.cuda.memory_allocated()/1e9:.2f} GB')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === persist: restore results/ from the shared backend =====================
# Each phase runs in a SEPARATE Colab Enterprise runtime, so /content starts
# empty. To see the previous phase's artifacts (discovery.json, the H6 jsonls,
# comparison.csv …) we restore results/ from a shared backend chosen here.
#
# RECOMMENDED on Colab Enterprise: a GCS bucket. Set it once per runtime:
#     %env GCS_BUCKET=gs://your-bucket-name
# (gsutil is pre-installed and the runtime service account has access.)
# Free-Colab fallback: Google Drive is auto-mounted if no bucket is set.
import os, subprocess
from src.persist import detect_backend, build_sync_cmd, remote_location

_drive_ok = Path('/content/drive').exists()
if not os.environ.get('GCS_BUCKET') and not _drive_ok:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        _drive_ok = Path('/content/drive').exists()
    except Exception as _e:
        print('Drive mount unavailable (fine if you are using GCS):', _e)

BACKEND = detect_backend(os.environ, _drive_ok)
BUCKET  = os.environ.get('GCS_BUCKET') or None
REMOTE  = remote_location(BACKEND, bucket=BUCKET) if BACKEND != 'local' else None
print(f'Persistence backend = {BACKEND}   remote = {REMOTE}')

import shutil as _shutil
def _sync(src, dst, backend, label):
    """Run one rsync/gsutil sync, guarding a missing CLI + first-phase noise."""
    cmd = build_sync_cmd(src, dst, backend)
    if _shutil.which(cmd[0]) is None:
        print(f'!! {cmd[0]!r} not on PATH — cannot {label}. '
              f'On Colab Enterprise gsutil is preinstalled; for Drive, rsync is.')
        return
    print(f'{label}:', ' '.join(cmd))
    # capture_output so an empty-remote gsutil CommandException on phase-0
    # restore doesn't dump a scary multi-line stderr; surface it only if it
    # looks like a real failure.
    r = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if r.returncode != 0 and 'does not name a directory' not in (r.stderr or ''):
        tail = (r.stderr or '').strip().splitlines()[-3:]
        if tail:
            print('   (note)', ' | '.join(tail))

RESULTS.mkdir(parents=True, exist_ok=True)
if REMOTE:
    if BACKEND == 'drive':
        Path(REMOTE).mkdir(parents=True, exist_ok=True)
    # remote -> local. check=False semantics: on the FIRST phase the remote is
    # empty, which is not an error.
    _sync(REMOTE, str(RESULTS), BACKEND, 'restore')
    print('Restored results/ from', REMOTE)
else:
    print('!! local backend: this phase will NOT see other phases\' outputs.')
    print('!! Set GCS_BUCKET (recommended) or mount Drive to chain phases.')

## 3. H1 — diagnosis-gate neuron

Contrastive set (pathognomonic vs ambiguous vignettes), gradient × activation discovery (Eqs. 2-4), multiplier sweep over top-5 candidates, capability check on a tiny MedQA set, activation-distribution plot.

In [ ]:
# === H1 — load data, discover ===
import traceback
try:

    from src.data import build_h1
    from src.discover import discover, sweep, best_multiplier, NeuronScore, DEFAULT_M_SWEEP

    H1_RESULTS = RESULTS / 'h1'; H1_RESULTS.mkdir(exist_ok=True)
    h1 = build_h1()
    print(f'Positive vignettes: {len(h1["positive"])}')
    print(f'Negative vignettes: {len(h1["negative"])}')

    cands_path = H1_RESULTS / 'candidates.json'
    if cands_path.exists():
        cands_raw = json.loads(cands_path.read_text())
        cands = [NeuronScore(**c) for c in cands_raw]
        print(f'Reloaded {len(cands)} candidates from {cands_path}')
    else:
        t0 = time.time()
        cands = discover(
            lm, positive_prompts=h1['positive'], negative_prompts=h1['negative'],
            target_phrases=h1['commitment_phrases'], icd10_tokens=h1['icd10_tokens'],
            layer_range=None, top_k=5,
        )
        print(f'Discovery done in {time.time()-t0:.1f}s')
        cands_path.write_text(json.dumps([c.__dict__ for c in cands], indent=2))

    print('\nTop-5 candidates:')
    for c in cands:
        print(f'  L{c.layer:>2}:F{c.neuron:<6}  score={c.score:+.4f}  a_pos={c.a_pos:+.3f}  a_neg={c.a_neg:+.3f}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H1 — multiplier sweep over top-5 + capability cost ===
import traceback
try:

    from types import SimpleNamespace
    from src.discover import best_multiplier_with_capability, mean_target_logprob_under, _target_first_token_ids, _mean_target_logprob

    sweep_path = H1_RESULTS / 'sweep.json'
    cap_path = H1_RESULTS / 'capability_sweep.json'
    probe = h1['positive'][:8]
    hard_for_capability = h1['hard_cases'][:8]  # cases the unmodified model handles

    if sweep_path.exists():
        sweep_raw = json.loads(sweep_path.read_text())
        print(f'Reloaded sweep ({len(sweep_raw)} rows)')
    else:
        sw = sweep(
            lm, candidates=cands, probes=probe,
            target_phrases=h1['commitment_phrases'], icd10_tokens=h1['icd10_tokens'],
            multipliers=DEFAULT_M_SWEEP, sample_prompt=h1['positive'][0],
        )
        sweep_raw = [s.__dict__ for s in sw]
        sweep_path.write_text(json.dumps(sweep_raw, indent=2))

    # Capability sweep: same (cand × m) grid, but evaluated on the HARD cases.
    # Drops in this log-prob = capability loss under the intervention.
    if cap_path.exists():
        capability_lp = {tuple(eval(k)): v for k, v in json.loads(cap_path.read_text()).items()}
        print(f'Reloaded capability sweep ({len(capability_lp)} entries)')
    else:
        capability_lp = mean_target_logprob_under(
            lm, candidates=cands, capability_prompts=hard_for_capability,
            target_phrases=h1['commitment_phrases'], icd10_tokens=h1['icd10_tokens'],
            multipliers=DEFAULT_M_SWEEP,
        )
        cap_path.write_text(json.dumps({str(k): v for k, v in capability_lp.items()}, indent=2))

    # Baseline (no intervention) capability log-prob.
    target_ids = _target_first_token_ids(lm.tokenizer, h1['commitment_phrases'], h1['icd10_tokens'])
    baseline_cap = _mean_target_logprob(lm, hard_for_capability, target_ids)
    print(f'Baseline capability log-prob (no intervention): {baseline_cap:+.4f}')

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    by_n_sup = {}
    by_n_cap = {}
    for s in sweep_raw:
        k = (s['layer'], s['neuron'])
        by_n_sup.setdefault(k, []).append((s['multiplier'], s['target_logprob']))
    for (L, N, m), lp in capability_lp.items():
        by_n_cap.setdefault((L, N), []).append((m, lp))
    for k, pts in by_n_sup.items():
        pts.sort(); xs, ys = zip(*pts); axes[0].plot(xs, ys, marker='o', label=f'L{k[0]}:F{k[1]}')
    for k, pts in by_n_cap.items():
        pts.sort(); xs, ys = zip(*pts); axes[1].plot(xs, ys, marker='o', label=f'L{k[0]}:F{k[1]}')
    axes[0].set_title('suppression on probe set (lower = more suppressed)')
    axes[1].set_title('capability on hard cases (lower = more capability lost)')
    for ax in axes:
        ax.set_xlabel('multiplier m'); ax.set_ylabel('mean target log-prob')
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(H1_RESULTS / 'sweep.png', dpi=140); plt.show()

    # Composite m* selection: suppression - lambda * capability_cost.
    sweep_objs = [SimpleNamespace(**s) for s in sweep_raw]
    L_star, N_star, m_star, composite = best_multiplier_with_capability(
        sweep_objs, capability_lp, baseline_cap, lambda_cap=1.0,
    )
    print(f'\nBest gate (capability-aware): L{L_star}:F{N_star} at m={m_star}')
    naive_L, naive_N, naive_m = best_multiplier(sweep_objs)
    print(f'  (naive max-suppression would have picked: L{naive_L}:F{naive_N} at m={naive_m})')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H1 — activation distribution at winning neuron ===
import traceback
try:

    import numpy as np
    from src.model import clear_h

    acts_pos, acts_neg = [], []
    for prompts, bucket in [(h1['positive'], acts_pos), (h1['negative'], acts_neg)]:
        for p in prompts:
            enc = lm.tokenizer(p, return_tensors='pt').to(lm.device)
            with torch.no_grad():
                lm.model(input_ids=enc.input_ids, use_cache=False)
            h = lm.layers[L_star].mlp._h[0, :, N_star].detach().float().cpu().numpy()
            bucket.extend(h.tolist())
            clear_h(lm.layers)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(acts_pos, bins=40, alpha=0.6, label='positive (committed)')
    ax.hist(acts_neg, bins=40, alpha=0.6, label='negative (hedging / generic)')
    ax.set_title(f'H1 L{L_star}:F{N_star} per-token activations')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(H1_RESULTS / 'activations.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H1 — hard-case capability check (anchor Eq. 7) ===
import traceback
try:

    from src.hooks import anchor_intervention, constant_intervention
    from src.eval import score_hedging

    # Use HARD diagnostic vignettes: messy, multi-finding cases that require
    # committing to a working diagnosis. Easy fact recall ("chambers of the heart")
    # doesn't test the diagnosis-gate behavior at all.
    HARD_CAPABILITY = h1['hard_cases'][:20]

    def gen(prompt, max_new=64):
        enc = lm.tokenizer(prompt, return_tensors='pt').to(lm.device)
        with torch.no_grad():
            out = lm.model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                    pad_token_id=lm.tokenizer.pad_token_id)
        clear_h(lm.layers)
        return lm.tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)

    best_cand = next(c for c in cands if c.layer == L_star and c.neuron == N_star)
    d = float(best_cand.a_pos - best_cand.a_neg) or 1e-3
    print(f'Anchor d = {d:.4f} | m* = {m_star}')

    capability = []
    for q in HARD_CAPABILITY:
        baseline = gen(q, 64)
        with constant_intervention(lm.layers, N_star, m_star, L_star):
            const = gen(q, 64)
        with anchor_intervention(lm.layers, N_star, m_star, d, L_star, k=1.0):
            anchor = gen(q, 64)
        bh = score_hedging(baseline); ch = score_hedging(const); ah = score_hedging(anchor)
        capability.append({
            'q': q, 'baseline': baseline, 'constant': const, 'anchor': anchor,
            'baseline_hedge': bh.is_hedging, 'constant_hedge': ch.is_hedging, 'anchor_hedge': ah.is_hedging,
        })

    # Summary: how often each mode hedges (we expect anchor to hedge more than baseline).
    def rate(key): return sum(int(r[key]) for r in capability) / max(1, len(capability))
    print(f'Hedge rate:  baseline={rate("baseline_hedge"):.2f}  constant={rate("constant_hedge"):.2f}  anchor={rate("anchor_hedge"):.2f}')

    for row in capability[:3]:
        print('\nCASE:', row['q'][:120], '...')
        print(' baseline:', row['baseline'][:200])
        print(' constant:', row['constant'][:200])
        print(' anchor  :', row['anchor'][:200])
    (H1_RESULTS / 'capability.json').write_text(json.dumps(capability, indent=2))
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === free VRAM after H1 ===
import traceback
try:

    # Drop H1 per-token activation tensors. Keep cands / L_star / N_star / m_star.
    try:
        del acts_pos, acts_neg, sw, sweep_objs, sweep_raw, capability_lp
    except NameError:
        pass
    _free_vram('after H1')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 4. H2 — disease-specific concept neurons

For each of {sepsis, T2DM, MI, pneumonia, asthma, depression}: rank top-3 MLP neurons by standardized margin, then amplify on benign prompts.

In [ ]:
# === H2 — build corpus, rank concept neurons ===
import traceback
try:

    from src.concept import DISEASE_KEYWORDS, amplification_matrix, rank_concept_neurons, ConceptNeuron
    from src.data import build_h2

    H2_RESULTS = RESULTS / 'h2'; H2_RESULTS.mkdir(exist_ok=True)
    h2 = build_h2(n_per_disease=200)
    print('Corpus sizes:')
    for k, v in h2.items():
        if k != '_benign_prompts':
            print(f'  {k:<12} pos={len(v["positive"])} neg={len(v["negative"])}')

    concept_path = H2_RESULTS / 'concept_neurons.json'
    if concept_path.exists():
        concepts = {k: [ConceptNeuron(**c) for c in v] for k, v in json.loads(concept_path.read_text()).items()}
        print('\nReloaded concept neurons.')
    else:
        concepts = {}
        for disease in ['sepsis', 't2dm', 'mi', 'pneumonia', 'asthma', 'depression']:
            print(f'\n== {disease} ==')
            pos = h2[disease]['positive'][:120]
            neg = h2[disease]['negative']
            top = rank_concept_neurons(lm, positive=pos, negative=neg, top_k=3, disease=disease)
            for c in top:
                print(f'  L{c.layer:>2}:F{c.neuron:<6}  margin={c.margin:+.3f}  mean_pos={c.mean_pos:+.3f}  mean_neg={c.mean_neg:+.3f}')
            concepts[disease] = top
        concept_path.write_text(json.dumps(
            {k: [c.__dict__ for c in v] for k, v in concepts.items()}, indent=2))
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H2 — amplification matrix (relative multipliers) ===
import traceback
try:

    # Multipliers are RELATIVE to each neuron's natural activation scale
    # (multiplier * max(|mean_pos|, |mean_neg|)). Absolute multipliers in the
    # 20-160 range previously saturated the residual stream and produced
    # token-degenerate output — see prior 0/4 injection rate.
    amp_path = H2_RESULTS / 'amplification.json'
    benign = h2['_benign_prompts']['positive'][:4]
    multipliers = [0.0, 1.0, 2.0, 4.0, 8.0]

    def _schema_ok(payload):
        # Reject prior-run dumps written with absolute multipliers (20/80/160).
        expected = set(multipliers)
        for rows in payload.values():
            seen = {r['multiplier'] for r in rows}
            if not expected.issubset(seen):
                return False
        return True

    if amp_path.exists():
        cached = json.loads(amp_path.read_text())
        if _schema_ok(cached):
            amp_results = cached
            print('Reloaded amplification.')
        else:
            print(f'Cached {amp_path} written with stale multipliers — recomputing.')
            amp_path.unlink()
            amp_results = None
    else:
        amp_results = None

    if amp_results is None:
        amp_results = {}
        for disease, neurons in concepts.items():
            c = neurons[0]
            scale = max(abs(c.mean_pos), abs(c.mean_neg), 1e-6)
            print(f'Amplifying {disease} via L{c.layer}:F{c.neuron} (scale={scale:.3f})')
            rows = amplification_matrix(
                lm, neuron=c, benign_prompts=benign, multipliers=multipliers,
                max_new_tokens=64, concept_keywords=DISEASE_KEYWORDS[disease],
                relative=True,
            )
            amp_results[disease] = [r.__dict__ for r in rows]
        amp_path.write_text(json.dumps(amp_results, indent=2))

    print()
    print(f'{"disease":<12} | ' + ' | '.join(f'm={m:>5}' for m in multipliers))
    print('-' * 60)
    for disease, rows in amp_results.items():
        by_m = {m: 0 for m in multipliers}
        total = {m: 0 for m in multipliers}
        for r in rows:
            by_m[r['multiplier']] += int(r['mentions_concept'])
            total[r['multiplier']] += 1
        row_str = ' | '.join(f'{by_m[m]:>2}/{total[m]:<2}' for m in multipliers)
        print(f'{disease:<12} | {row_str}')

    print('\nSample generations at m =', max(multipliers))
    for disease, rows in amp_results.items():
        sample = next((r for r in rows if r['multiplier'] == max(multipliers)), None)
        if sample:
            print(f'\n  {disease} L{sample["layer"]}:F{sample["neuron"]} on "{sample["prompt"][:40]}..."')
            print(f'    -> {sample["generation"][:200]}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 5. H3 — symptom→diagnosis routing

Per-layer residual-stream activation patching across clean/corrupted vignette pairs, then drill into the critical layer at the MLP-neuron level.

In [ ]:
# === H3 — verify diagnosis tokens ===
import traceback
try:

    from src.data import H3_PAIRS, verify_h3_tokens
    from src.patching import patch_layers, patch_neurons_at_layer

    H3_RESULTS = RESULTS / 'h3'; H3_RESULTS.mkdir(exist_ok=True)
    for label, tid in verify_h3_tokens(lm.tokenizer):
        print(f'  {label:<12} -> token id {tid}  =  {lm.tokenizer.decode([tid])!r}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H3 — per-layer patching curve ===
import traceback
try:

    patch_path = H3_RESULTS / 'patch_layers.json'
    if patch_path.exists():
        patch_data = json.loads(patch_path.read_text())
        print('Reloaded patch data.')
    else:
        patch_data = {}
        for pair in H3_PAIRS:
            print(f'Patching pair {pair.pair_id} ...')
            scores = patch_layers(
                lm, pair.clean_prompt, pair.corrupted_prompt,
                pair.clean_dx, pair.corrupted_dx, pair.pair_id,
            )
            patch_data[pair.pair_id] = [s.__dict__ for s in scores]
        patch_path.write_text(json.dumps(patch_data, indent=2))

    import matplotlib.pyplot as plt, numpy as np
    fig, ax = plt.subplots(figsize=(9, 5))
    for pid, rows in patch_data.items():
        xs = [r['layer'] for r in rows]
        ys = [r['score'] for r in rows]
        ax.plot(xs, ys, marker='o', label=pid, alpha=0.75)
    ax.axhline(0, color='k', lw=0.5); ax.axhline(1, color='g', lw=0.5, ls='--')
    ax.set_xlabel('layer'); ax.set_ylabel('(patched − corrupt) / (clean − corrupt)')
    ax.set_title('H3 — per-layer residual patching')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(H3_RESULTS / 'patch_layers.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H3 — full d_ff drill at critical layer ===
import traceback
try:

    mean_per_layer = {}
    for L in range(lm.n_layers):
        vals = [r['score'] for rows in patch_data.values() for r in rows if r['layer'] == L]
        if vals:
            mean_per_layer[L] = float(np.mean(vals))
    critical = max(mean_per_layer, key=mean_per_layer.get)
    print(f'Critical layer (mean score {mean_per_layer[critical]:+.3f}): L{critical}')

    # FULL-d_ff drill at the critical layer — every neuron gets patched in turn.
    # For an 8B-class model (d_ff≈14k) that's ~14k forwards through one pair on the
    # Blackwell GPU, ~30 min wall. Output is the per-neuron routing contribution.
    drill_path = H3_RESULTS / f'drill_L{critical}_full.json'
    if drill_path.exists():
        drill_data = json.loads(drill_path.read_text())
        print(f'Reloaded full drill ({len(drill_data)} neurons).')
    else:
        pair = H3_PAIRS[0]
        print(f'Drilling full d_ff={lm.d_ff} at L{critical} on pair {pair.pair_id} ...')
        drill = patch_neurons_at_layer(
            lm, pair.clean_prompt, pair.corrupted_prompt,
            pair.clean_dx, pair.corrupted_dx, layer_idx=critical,
            neuron_indices=list(range(lm.d_ff)),
            pair_id=pair.pair_id,
        )
        drill_data = [d.__dict__ for d in drill]
        drill_path.write_text(json.dumps(drill_data, indent=2))

    top10 = sorted(drill_data, key=lambda d: abs(d['score']), reverse=True)[:10]
    print(f'\nTop-10 neurons at L{critical} by |score|:')
    for d in top10:
        print(f'  L{d["layer"]}:F{d["neuron"]:<6}  score={d["score"]:+.3f}')

    if len(patch_data) > 1:
        pairs = list(patch_data.keys())
        layers = sorted({r['layer'] for rows in patch_data.values() for r in rows})
        mat = np.array([[next((r['score'] for r in patch_data[p] if r['layer'] == L), 0.0) for L in layers] for p in pairs])
        fig, ax = plt.subplots(figsize=(10, 0.5 + 0.4 * len(pairs)))
        im = ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_yticks(range(len(pairs)), pairs)
        tick_idx = range(0, len(layers), max(1, len(layers) // 10))
        ax.set_xticks(list(tick_idx), [layers[i] for i in tick_idx])
        ax.set_xlabel('layer'); plt.colorbar(im, ax=ax)
        plt.title('H3 patching heatmap (pair × layer)')
        plt.tight_layout(); plt.savefig(H3_RESULTS / 'heatmap.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === free VRAM after H3 drill ===
import traceback
try:

    # H3 full-d_ff drill holds 14k×per-pair scores in memory. Drop them; we only
    # need `critical` and `mean_per_layer` downstream.
    try:
        del drill, drill_data, mat
    except NameError:
        pass
    _free_vram('after H3')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 6. H4 — hallucination / false-confidence neurons

We push the model past its knowledge limit with a *trap* set: under-specified vignettes (single sign, no workup), contradictory findings, rare/exotic real diseases, and **fabricated syndromes** (controls — any commitment is hallucination by construction). A clinician would refuse or ask for more info; the model usually commits anyway. The neurons that fire on **trap-committed** prompts but stay silent on **hedged** prompts isolate the commitment gate. Subtracting the **pathognomonic-committed** activation map leaves the *false-confidence* component: neurons that fire harder when the knowledge is insufficient than when it is solid.

In [ ]:
# === H4 — classify + find hallucination neurons ===
import traceback
try:

    from src.data import build_h4
    from src.hallucinate import find_hallucination_neurons, HallucinationNeuron

    H4_RESULTS = RESULTS / 'h4'; H4_RESULTS.mkdir(exist_ok=True)
    h4 = build_h4()
    print(f'Trap set: {len(h4["trap"])} prompts')
    print(f'Pathognomonic: {len(h4["pathognomonic"])} prompts')

    halluc_path = H4_RESULTS / 'hallucination_neurons.json'
    classif_path = H4_RESULTS / 'classifications.json'
    if halluc_path.exists():
        halluc_neurons = [HallucinationNeuron(**c) for c in json.loads(halluc_path.read_text())]
        classifications = json.loads(classif_path.read_text())
        print(f'Reloaded {len(halluc_neurons)} hallucination neurons')
    else:
        halluc_neurons, classifications = find_hallucination_neurons(
            lm,
            trap_prompts=h4['trap'],
            pathognomonic_prompts=h4['pathognomonic'][:10],     # representative pathognomonic
            hedge_prompts=h1['negative'][:10],                  # ambiguous control
            target_phrases=h4['commitment_phrases'],
            icd10_tokens=h4['icd10_tokens'],
            layer_range=None,  # auto: layers >= n_layers // 3
            top_k=10,
            commit_p_threshold=0.10,
        )
        halluc_path.write_text(json.dumps([n.__dict__ for n in halluc_neurons], indent=2))
        classif_path.write_text(json.dumps(classifications, indent=2))

    # How often did the model commit when it shouldn't have?
    def commit_rate(bucket):
        rows = classifications[bucket]
        return sum(1 for _, c, _ in rows if c) / max(1, len(rows))

    print()
    print(f'Commit rate on trap  (should be ~0 ideally): {commit_rate("trap"):.2f}')
    print(f'Commit rate on pathognomonic (should be high): {commit_rate("pathognomonic"):.2f}')
    print(f'Commit rate on hedge (should be ~0):          {commit_rate("hedge"):.2f}')

    # Show committed traps (these are the hallucinations).
    print('\nTrap prompts the model COMMITTED to (hallucinations):')
    for p, committed, gen in classifications['trap']:
        if committed:
            print(f'  Q: {p[:90]}...')
            print(f'     -> {gen[:140]}')

    print('\nTop hallucination neurons (delta = a_trap_commit - a_pathognomonic):')
    for n in halluc_neurons:
        print(f'  L{n.layer:>2}:F{n.neuron:<6}  delta={n.delta:+.3f}  '
              f'a_trap={n.a_trap:+.3f}  a_pathog={n.a_pathog:+.3f}  a_hedge={n.a_hedge:+.3f}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H4 — layer profile of hallucination signal ===
import traceback
try:

    # Aggregate delta by layer to see WHERE the false-confidence signal lives.
    import numpy as np, matplotlib.pyplot as plt
    by_layer = {}
    for n in halluc_neurons:
        by_layer.setdefault(n.layer, []).append(n.delta)
    layers = sorted(by_layer)
    mean_delta = [float(np.mean(by_layer[L])) for L in layers]
    max_delta = [float(np.max(by_layer[L])) for L in layers]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(layers, mean_delta, marker='o', label='mean delta (top-10 per layer)')
    ax.plot(layers, max_delta, marker='s', label='max delta')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('layer'); ax.set_ylabel('a_trap_commit - a_pathognomonic_commit')
    ax.set_title('H4 — false-confidence signal by layer')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(H4_RESULTS / 'layer_profile.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === free VRAM after H4 ===
import traceback
try:

    # H4 captured per-bucket signed-max activations. Drop them before H5/H6.
    try:
        del a_trap, a_pathog, a_hedge
    except NameError:
        pass
    _free_vram('after H4')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 11. H4-extended — per-category hallucination commit rates

The trap DB is now ~50 prompts across categories: underspecified, contradictory, rare, fabricated, impossible. Per-category commit rates isolate *which kind* of hallucination dominates. The most diagnostic are `fabricated` and `impossible` — any commitment is by construction wrong.

In [ ]:
# === H4 — per-category commit rates from cached classifications ===
import traceback
try:

    import collections, json as _json

    # We have the trap classifications saved by H4 (cell 22).
    classif = _json.loads((H4_RESULTS / 'classifications.json').read_text())
    # `classifications['trap']` is a list of [prompt, committed, generation].
    from src.data import TRAP_DB
    cat_of = dict(TRAP_DB)   # prompt -> category

    counts = collections.defaultdict(lambda: [0, 0])
    for prompt, committed, gen in classif.get('trap', []):
        cat = cat_of.get(prompt, 'unknown')
        counts[cat][0] += int(bool(committed))
        counts[cat][1] += 1

    print(f'{"category":<16}  {"commit_rate":>12}  {"n":>4}')
    print('-' * 38)
    for cat in ['underspecified', 'contradictory', 'rare', 'fabricated', 'impossible', 'unknown']:
        c, n = counts.get(cat, [0, 0])
        if n:
            print(f'{cat:<16}  {c/n:>11.2f}  {n:>4}   ({c} committed)')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## 7. H5 — overconfidence / miscalibration neurons

Distinct from H4 (committed when it should have refused). H5 targets a subtler failure: cases where the model's **next-token probability** of its own diagnosis is low (it doesn't really know), but when asked *'Are you confident?'* it says **yes** with high probability.

Per case we measure:
- `p_dx` = model's max softmax probability on the diagnosis slot (actual top-1 confidence)
- `p_yes` = probability mass on confident-attestation tokens ("Yes", "Sure", ...) on the follow-up prompt
- `calibration_gap = p_yes − p_dx`

Then we Pearson-correlate every MLP neuron's attestation-time activation with `calibration_gap` across the hard-case set. Neurons with high positive correlation fire harder when the model is *more* overconfident than warranted — they encode 'I am sure' independent of whether the underlying answer is well-supported.

In [ ]:
# === H5 — measure calibration on hard cases + rank neurons ===
import traceback
try:

    from src.calibration import find_overconfidence_neurons, CalibrationCase, OverconfidenceNeuron

    H5_RESULTS = RESULTS / 'h5'; H5_RESULTS.mkdir(exist_ok=True)
    h5_cases_path = H5_RESULTS / 'cases.json'
    h5_neurons_path = H5_RESULTS / 'overconfidence_neurons.json'

    if h5_cases_path.exists() and h5_neurons_path.exists():
        cases_dump = json.loads(h5_cases_path.read_text())
        over_neurons = [OverconfidenceNeuron(**n) for n in json.loads(h5_neurons_path.read_text())]
        print(f'Reloaded H5: {len(cases_dump)} cases, {len(over_neurons)} neurons.')
    else:
        hard_for_h5 = h1['hard_cases']  # 20 messy multi-finding vignettes
        cases, over_neurons = find_overconfidence_neurons(
            lm, hard_cases=hard_for_h5,
            layer_range=None,           # default = later half (commitment / confidence)
            top_k=15, overconf_threshold=0.3, gap_high_low_n=4,
        )
        # Persist (drop the per-layer activation tensors — too big for JSON).
        cases_dump = [
            {
                'case': c.case, 'dx_text': c.dx_text,
                'p_dx': c.p_dx, 'p_yes': c.p_yes, 'p_no': c.p_no,
                'calibration_gap': c.calibration_gap,
            }
            for c in cases
        ]
        h5_cases_path.write_text(json.dumps(cases_dump, indent=2))
        h5_neurons_path.write_text(json.dumps([n.__dict__ for n in over_neurons], indent=2))

    # Calibration table.
    print(f'{"#":>3}  {"p_dx":>6}  {"p_yes":>6}  {"p_no":>6}  {"gap":>7}  dx -> case-prefix')
    print('-' * 110)
    for i, c in enumerate(sorted(cases_dump, key=lambda x: x['calibration_gap'], reverse=True)):
        case_prefix = c['case'][:55].replace(chr(10), ' ')
        dx = c['dx_text'][:32].replace(chr(10), ' ')
        print(f'{i:>3}  {c["p_dx"]:>6.3f}  {c["p_yes"]:>6.3f}  {c["p_no"]:>6.3f}  {c["calibration_gap"]:>+7.3f}  {dx:<32} | {case_prefix}...')

    print('\nTop overconfidence neurons (corr(activation, calibration_gap)):')
    for n in over_neurons:
        print(f'  L{n.layer:>2}:F{n.neuron:<6}  r={n.pearson_r:+.3f}  '
              f'mean_overconf={n.mean_act_overconf:+.3f}  mean_calib={n.mean_act_calib:+.3f}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H5 — plot calibration scatter + neuron correlation ===
import traceback
try:

    import numpy as np, matplotlib.pyplot as plt
    p_dx = np.array([c['p_dx'] for c in cases_dump])
    p_yes = np.array([c['p_yes'] for c in cases_dump])
    gap = np.array([c['calibration_gap'] for c in cases_dump])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(p_dx, p_yes, alpha=0.7)
    axes[0].plot([0, 1], [0, 1], 'k--', lw=0.5, label='perfect calibration')
    axes[0].set_xlabel('p_dx (actual top-1 confidence)')
    axes[0].set_ylabel('p_yes (stated confidence)')
    axes[0].set_title('H5 — calibration scatter (points above y=x are overconfident)')
    axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1); axes[0].grid(alpha=0.3); axes[0].legend()

    # Per-layer mean pearson_r of the top neurons.
    by_layer = {}
    for n in over_neurons:
        by_layer.setdefault(n.layer, []).append(n.pearson_r)
    layers = sorted(by_layer)
    mean_r = [np.mean(by_layer[L]) for L in layers]
    axes[1].bar(layers, mean_r, alpha=0.7)
    axes[1].axhline(0, color='k', lw=0.5)
    axes[1].set_xlabel('layer'); axes[1].set_ylabel('mean Pearson r (top overconf neurons)')
    axes[1].set_title('H5 — overconfidence signal by layer')
    axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(H5_RESULTS / 'calibration.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

## Save discovery checkpoint + mirror results

In [ ]:
# === save discovery checkpoint (read by 02/03/04) ==========================
import time
from src.checkpoint import Discovery, save_discovery

# Re-derive the anchor d from the H1 candidate (robust to the `d` global being
# shadowed by H3's drill loop — same guard the monolith H6 cell uses).
_best = next(c for c in cands if c.layer == L_star and c.neuron == N_star)
_anchor_d = float(_best.a_pos - _best.a_neg) or 1e-3

disc = Discovery(
    model_name=MODEL_NAME,
    gate_layer=int(L_star), gate_neuron=int(N_star),
    gate_m_star=float(m_star), gate_anchor_d=float(_anchor_d),
    critical_layer=int(critical),
    layer_scores=[float(mean_per_layer[k]) for k in sorted(mean_per_layer)],
    halluc_neurons=[{'layer': int(n.layer), 'neuron': int(n.neuron)} for n in halluc_neurons[:3]],
    overconf_neurons=[{'layer': int(n.layer), 'neuron': int(n.neuron)} for n in over_neurons[:3]],
    git_sha=sha, created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    n_layers=int(lm.n_layers), d_ff=int(lm.d_ff),
    extra={'mean_per_layer': {str(k): float(v) for k, v in mean_per_layer.items()}},
)
save_discovery(RESULTS, disc)
print('Saved', RESULTS / 'discovery.json')
print('  gate        :', disc.gate_dict())
print('  critical    :', disc.critical_layer)
print('  halluc top3 :', disc.halluc_neurons)
print('  overconf t3 :', disc.overconf_neurons)

In [ ]:
# === persist: mirror results/ back to the shared backend ===================
# Run this LAST so the next phase's runtime can restore what this phase made.
# (`_sync` was defined in the restore cell — same guards apply.)
if REMOTE:
    if BACKEND == 'drive':
        Path(REMOTE).mkdir(parents=True, exist_ok=True)
    _sync(str(RESULTS), REMOTE, BACKEND, 'mirror')       # local -> remote
    print('Mirrored results/ →', REMOTE)
else:
    print('local backend — results stay in /content/results only this session.')